# PostgreSQL Database Demo

This notebook provides guidance on data operations with PostgreSQL.

> **Note**: Table structure has been automatically initialized by Docker from the `init.sql` file when you run `docker-compose up`.

## 1. Connection

Establish a connection to PostgreSQL and check existing tables.

In [ ]:
import psycopg2
import pandas as pd
from datetime import datetime

# Connection information (Match with docker-compose.yml)
conn_params = {
    "host": "127.0.0.1",
    "port": 5433,
    "database": "postgres",
    "user": "postgres",
    "password": "yourpassword"
}

try:
    conn = psycopg2.connect(**conn_params)
    cur = conn.cursor()
    
    # Check if the klines table exists
    cur.execute("SELECT EXISTS (SELECT FROM information_schema.tables WHERE table_name = 'klines');")
    exists = cur.fetchone()[0]
    
    if exists:
        print("✅ Connection successful! Table 'klines' is ready.")
    else:
        print("⚠️ Connection successful but table 'klines' not found. Please check init.sql file.")
        
except Exception as e:
    print(f"❌ Connection error: {e}")

✅ Kết nối thành công! Bảng 'klines' đã sẵn sàng.


## 2. Write Commands

Add sample data to PostgreSQL.

In [ ]:
import time

# Start measuring execution time
start_time = time.perf_counter()

symbol = 'ETHUSDT'
interval = '1h'
now = datetime.now()
ts = int(now.timestamp() * 1000)

insert_query = """
INSERT INTO klines (symbol, interval, date_bucket, timestamp, open, high, low, close, volume)
VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s)
ON CONFLICT (symbol, interval, date_bucket, timestamp) DO UPDATE 
SET close = EXCLUDED.close;
"""

# Execute the command
cur.execute(insert_query, (
    symbol, interval, now.date(), ts, 
    '3500.0', '3550.0', '3480.0', '3520.5', '10.5'
))
conn.commit()

# Calculate elapsed time (convert to milliseconds)
execution_time = (time.perf_counter() - start_time) * 1000

print(f"✅ Data saved for {symbol} - {execution_time:.2f} ms")

✅ Đã lưu dữ liệu cho ETHUSDT - 12.68 ms


## 3. Query Commands

Query data and display as a DataFrame.

In [ ]:
start_time = time.perf_counter()

cur.execute("SELECT * FROM klines WHERE symbol = %s LIMIT 5", (symbol,))
rows = cur.fetchall()
colnames = [desc[0] for desc in cur.description]

execution_time = (time.perf_counter() - start_time) * 1000

print(f"Speed: {execution_time:.2f} ms")
df = pd.DataFrame(rows, columns=colnames)
df.head()

Speed: 2.19 ms


,symbol,interval,date_bucket,timestamp,open,high,low,close,volume
0,ETHUSDT,1h,2026-05-07,1778097889105,3500.0,3550.0,3480.0,3520.5,10.5
1,ETHUSDT,1h,2026-05-07,1778162530694,3500.0,3550.0,3480.0,3520.5,10.5
